# 3. PLS regression coefficients and score plot (Figures 1 and 2)

| Output | Corresponding item in the paper |
|---|---|
| `output/figure_2_coefficients.png` | Figure 2 |
| `output/figure_1a_score_plot.png` | Figure 1, score plot (left panel) |
| `output/pls_coefficients.csv` | the full list of coefficients |

**Input:** `output/pls_model.joblib` (run `0_fingerprint_and_pls.ipynb` first),
`data/maccskeys_meaning.csv`, `data/solvent.csv`.

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# Resolve paths relative to the repository root, so that the notebook runs
# both from notebooks/ and from the repository root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
QM = ROOT / "qm" / "qm_nbo_t6311++g"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

SAVE_DPI = 600        # resolution of the saved figure files
plt.rcParams["figure.dpi"] = 100      # on-screen resolution; keeps the notebook fast
plt.rcParams["text.parse_math"] = False   # substructure names may contain "$"

bundle = joblib.load(OUT / "pls_model.joblib")
model = bundle["model"]

# maccskeys_meaning.csv describes bits 1-166; bit 0 is the padding bit and is
# dropped from every per-bit quantity with [1:]
maccs_keys = pd.read_csv(DATA / "maccskeys_meaning.csv")
print(f"{len(maccs_keys)} MACCS bits described, "
      f"{bundle['n_components']} latent variables in the model")

## Regression coefficients

`model.coef_` has shape `(n_targets, n_features)` in scikit-learn 1.1 and later, so
`coef_[0]` is the vector of coefficients for the single response. The leading element
corresponds to the padding bit and is discarded.

A positive coefficient means that the presence of the substructure is associated with
a higher predicted yield.

In [ ]:
coefficients = model.coef_[0][1:]
assert len(coefficients) == len(maccs_keys), "bit count and description file disagree"

maccs_keys["coef"] = coefficients
maccs_keys["coef_abs"] = np.abs(coefficients)
maccs_keys = maccs_keys.sort_values("coef_abs", ascending=False)

maccs_keys.to_csv(OUT / "pls_coefficients.csv", index=False)
maccs_keys.head(10)

## Figure 2: the ten largest coefficients

Positive and negative coefficients are drawn in different colours, and the tick labels
are coloured to match so that the sign remains readable when the figure is reduced in
size.

In [ ]:
POSITIVE_COLOR = "#FF2E62"
NEGATIVE_COLOR = "#232A34"
N_SHOWN = 10

top = maccs_keys.head(N_SHOWN)
colors = [POSITIVE_COLOR if c > 0 else NEGATIVE_COLOR for c in top["coef"]]

fig, ax = plt.subplots(figsize=(12, 8))
ax.bar(top["subscription"], top["coef"], color=colors)
ax.set_xlabel("MACCS Keys fingerprint", fontsize=18, labelpad=15)
ax.set_ylabel("Coefficient", fontsize=18, labelpad=15)
ax.tick_params(axis="x", rotation=90, labelsize=18)
for label, color in zip(ax.get_xticklabels(), colors):
    label.set_color(color)

fig.savefig(OUT / "figure_2_coefficients.png", dpi=SAVE_DPI, bbox_inches="tight")
plt.show()

## Figure 1 (left): score plot

`model.x_scores_` gives the position of each **training solvent** in the space spanned
by the latent variables. Points are coloured by the observed yield, which makes the
separation along LV1 visible.

The axes are labelled LV1 and LV2 rather than PC1 and PC2: PLS determines these
directions using the response variable, so they are not the principal components of an
unsupervised principal component analysis.

In [ ]:
train = pd.read_csv(DATA / "solvent.csv").dropna(subset=["exp_yield"]).reset_index(drop=True)
scores = model.x_scores_
assert len(train) == len(scores), "solvent.csv does not match the stored model"

fig, ax = plt.subplots(figsize=(4, 4))
sc = ax.scatter(scores[:, 0], scores[:, 1], c=train["exp_yield"], cmap="coolwarm",
                vmin=0, vmax=100, zorder=3)
for (x, y), name in zip(scores[:, :2], train["name"]):
    ax.annotate(name, (x, y), fontsize=8, xytext=(3, 3), textcoords="offset points")

ax.axhline(0, color="grey", lw=1)
ax.axvline(0, color="grey", lw=1)
ax.set(xlim=(-8, 8), ylim=(-8, 8), xlabel="LV1", ylabel="LV2")
ax.set_xticks(np.arange(-8, 9, 2))
ax.set_yticks(np.arange(-8, 9, 2))
ax.set_aspect(1, adjustable="box")
fig.colorbar(sc, ax=ax, label="Experimental yield [%]")

fig.savefig(OUT / "figure_1a_score_plot.png", dpi=SAVE_DPI, bbox_inches="tight")
plt.show()

High-yielding solvents lie on the positive side of LV1 and low-yielding solvents on
the negative side. Because LV1 was constructed using the yields, this separation is
built into the method; the informative part is *which structural features* load onto
LV1, which is the subject of `4_pls_loadings.ipynb`.